In [12]:
import tensorflow as tf
from transformers import AutoTokenizer, TFAutoModel, AutoModel
from tensorflow.keras.layers import Input, Bidirectional, LSTM, Dropout, Flatten, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split
import pandas as pd
import torch
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import os



In [23]:


from transformers import AutoTokenizer, AutoModel

tokenizer = AutoTokenizer.from_pretrained(
    '/kaggle/input/base-model-indobertweet/transformers/default/1/base-model-indobertweet'
)
model = AutoModel.from_pretrained("/kaggle/input/base-model-indobertweet/transformers/default/1/base-model-indobertweet")

In [14]:
# List available physical GPUs
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Enable dynamic memory growth (prevents TensorFlow from reserving all VRAM)
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"✅ Using GPU: {gpus[0].name}")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ No GPU detected, training will use CPU instead.")

✅ Using GPU: /physical_device:GPU:0


In [24]:
# =============================================
# IndoBERTweet + BiLSTM for Binary Classification (PyTorch)
# =============================================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import numpy as np

# -------------------------------
# 1. Load dataset
# -------------------------------
df = pd.read_json('/kaggle/input/fetched-data-final/fetched_data_final_dedup.json')
X = df['text'].tolist()
y = df['label'].astype(int).tolist()

# -------------------------------
# 2. Load IndoBERTweet
# -------------------------------
bert_model = model

# -------------------------------
# 3. Configuration
# -------------------------------
max_length = 128
batch_size = 16
epochs = 20
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

# -------------------------------
# 4. Dataset class
# -------------------------------
class IndoDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.float)
        }

# -------------------------------
# 5. Train-test split
# -------------------------------
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

train_dataset = IndoDataset(X_train, y_train, tokenizer, max_length)
test_dataset = IndoDataset(X_test, y_test, tokenizer, max_length)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

# -------------------------------
# 6. Define model (IndoBERTweet + BiLSTM)
# -------------------------------
class IndoBERTweetBiLSTM(nn.Module):
    def __init__(self, bert_model):
        super(IndoBERTweetBiLSTM, self).__init__()
        self.bert = bert_model
        self.lstm = nn.LSTM(input_size=768, hidden_size=64, num_layers=1, 
                            batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(64 * 2 * max_length, 1)  # Flattened features
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():  # Freeze IndoBERTweet (feature-based)
            outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
            last_hidden_state = outputs.last_hidden_state
        lstm_out, _ = self.lstm(last_hidden_state)
        x = self.dropout(lstm_out)
        x = x.contiguous().view(x.size(0), -1)  # Flatten
        x = self.fc(x)
        return self.sigmoid(x)

model = IndoBERTweetBiLSTM(bert_model).to(device)

# -------------------------------
# 7. Define optimizer and loss
# -------------------------------
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=2e-5)
best_val_loss = float('inf')
patience = 3  # stop if no improvement after 3 epochs
patience_counter = 0
best_model_path = "/kaggle/working/best_indobertweet_bilstm_model.pt"

# -------------------------------
# 8. Training loop
# -------------------------------
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)
    # print(f"Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}")

    # -------------------------------
    # Validation
    # -------------------------------
    model.eval()
    val_loss = 0.0
    y_true, y_pred = [], []

    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].unsqueeze(1).to(device)

            outputs = model(input_ids, attention_mask)
            loss = criterion(outputs, labels)
            val_loss += loss.item()

            preds = (outputs > 0.5).float().cpu().numpy()
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds)

    avg_val_loss = val_loss / len(test_loader)
    val_acc = accuracy_score(y_true, y_pred)

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {avg_train_loss:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # -------------------------------
    # Early Stopping logic
    # -------------------------------
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"✅ Model improved and saved at epoch {epoch+1}")
    else:
        patience_counter += 1
        print(f"⚠️ No improvement for {patience_counter} epoch(s)")
        if patience_counter >= patience:
            print("⛔ Early stopping triggered.")
            break


✅ Using device: cuda
Epoch [1/20] | Train Loss: 0.4294 | Val Loss: 0.3225 | Val Acc: 0.8653
✅ Model improved and saved at epoch 1
Epoch [2/20] | Train Loss: 0.2920 | Val Loss: 0.2825 | Val Acc: 0.8875
✅ Model improved and saved at epoch 2
Epoch [3/20] | Train Loss: 0.2553 | Val Loss: 0.2709 | Val Acc: 0.8963
✅ Model improved and saved at epoch 3
Epoch [4/20] | Train Loss: 0.2345 | Val Loss: 0.2585 | Val Acc: 0.9041
✅ Model improved and saved at epoch 4
Epoch [5/20] | Train Loss: 0.2233 | Val Loss: 0.2511 | Val Acc: 0.9063
✅ Model improved and saved at epoch 5
Epoch [6/20] | Train Loss: 0.2124 | Val Loss: 0.2475 | Val Acc: 0.9089
✅ Model improved and saved at epoch 6
Epoch [7/20] | Train Loss: 0.2033 | Val Loss: 0.2428 | Val Acc: 0.9085
✅ Model improved and saved at epoch 7
Epoch [8/20] | Train Loss: 0.1938 | Val Loss: 0.2424 | Val Acc: 0.9071
✅ Model improved and saved at epoch 8
Epoch [9/20] | Train Loss: 0.1889 | Val Loss: 0.2349 | Val Acc: 0.9133
✅ Model improved and saved at epoch 

In [25]:
# -------------------------------
# 9. Evaluation
# -------------------------------
model.eval()

y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids, attention_mask)
        preds = (outputs > 0.5).float().cpu().numpy()
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds)

print("\nClassification Report:")
print(classification_report(y_true, y_pred, digits=4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))
print(f"\n✅ Test Accuracy: {accuracy_score(y_true, y_pred):.4f}")

# -------------------------------
# 10. Save model
# -------------------------------
# torch.save(model.state_dict(), "/kaggle/working/indobertweet_bilstm_model.pt")
# print("\n💾 Model saved successfully!")



Classification Report:
              precision    recall  f1-score   support

         0.0     0.9054    0.9321    0.9185      1119
         1.0     0.9335    0.9072    0.9202      1175

    accuracy                         0.9194      2294
   macro avg     0.9194    0.9197    0.9193      2294
weighted avg     0.9198    0.9194    0.9194      2294


Confusion Matrix:
[[1043   76]
 [ 109 1066]]

✅ Test Accuracy: 0.9194


In [26]:
from sklearn.metrics import classification_report, confusion_matrix

# Predictions
y_pred = (model.predict([X_test_input_ids, X_test_attention]) > 0.5).astype("int32")

# Reports
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

AttributeError: 'IndoBERTweetBiLSTM' object has no attribute 'predict'